# H1 ABSA-RTS Pipeline (Colab)

This notebook runs the **H1 negation/contrast diagnostics** pipeline end-to-end:
1. Build ABSA-RTS pairs/eval set
2. Generate A1 prediction file (placeholder predictor with configurable noise)
3. Evaluate A0 vs A1 and export metrics


## 0) Set project path

- If you cloned the repository in Colab, set `PROJECT_ROOT` accordingly.
- If your repository is in Google Drive, mount Drive first and point `PROJECT_ROOT` to that folder.


In [ ]:
# Optional: mount Drive if your repo is stored there
# from google.colab import drive
# drive.mount('/content/drive')

import os
import sys
from pathlib import Path

# Update this path to your repository root in Colab
PROJECT_ROOT = Path('/content/AutoResearch-Copilot')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Update the path in this cell.'
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Working directory:', PROJECT_ROOT)


## 1) Define paths

In [ ]:
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'h1-negation-contrast-diagnostics' / 'results'
SEED_CSV = PROJECT_ROOT / 'data' / 'absa_rts_seed_restaurants.csv'
EVAL_CSV = RESULTS_DIR / 'absa_rts_eval.csv'
PAIRS_CSV = RESULTS_DIR / 'absa_rts_pairs.csv'
A1_PRED_CSV = RESULTS_DIR / 'a1_predictions.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)
print('Seed csv:', SEED_CSV)


## 2) Build ABSA-RTS (negation + contrast)

In [ ]:
!PYTHONPATH=src python -m absa_rts.build_rts \
  --input data/absa_rts_seed_restaurants.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results \
  --domain restaurants

## 3) Generate A1 predictions (replace later with your real model outputs)

This uses a placeholder predictor + noise to simulate an imperfect baseline file.
Set `NOISE=0.0` if you want deterministic predictions from the simple heuristic.

In [ ]:
NOISE = 0.10
SEED = 17

!PYTHONPATH=src python -m absa_rts.generate_predictions \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --output-csv experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --noise {NOISE} \
  --seed {SEED}

## 4) Evaluate A0 vs A1

In [ ]:
!PYTHONPATH=src python -m absa_rts.evaluate_baselines \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --pairs-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_pairs.csv \
  --a1-predictions experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results

## 5) Inspect outputs

In [ ]:
import json
from pathlib import Path

summary_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_summary.csv')
metrics_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_metrics.json')

print(summary_path.read_text(encoding='utf-8'))
print('\nDetailed metrics:\n')
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2))


## 6) (Optional) Export result files to Drive

Uncomment and set your destination path if needed.

In [ ]:
# import shutil
# target = Path('/content/drive/MyDrive/absa_h1_results')
# target.mkdir(parents=True, exist_ok=True)
# for p in RESULTS_DIR.glob('*'):
#     if p.is_file():
#         shutil.copy2(p, target / p.name)
# print('Copied files to', target)
